# 02 SQL Business Queries

## 2.1 Business Objective

This notebook uses SQL to transform cleaned product sales data into business summary tables for management review.

The goal is to answer five business questions:

1. Which SKUs contribute the most revenue?
2. Which SKUs sell the most units?
3. Which SKUs are low-volume or long-tail products?
4. Which countries or regions contribute the most demand?
5. How do monthly sales volume and revenue change over time?

SQL is used in this step because business teams often rely on structured queries to generate consistent reporting tables from transaction-level data.

## 2.2 Data Foundation from 01

This notebook uses `clean_sales.csv`, which was created in `01_data_cleaning.ipynb`.

The cleaned sales dataset includes valid physical product sales only. The following records were excluded or separated in the data cleaning stage:

1. Cancellations and returns.
2. Duplicate rows.
3. Records with non-positive quantity or unit price.
4. Records without product descriptions.
5. Non-product transaction lines such as postage, fees, bank charges, discounts, and manual adjustments.

After the updated cleaning step, the main product sales dataset contains 522,716 valid product sales rows.

This ensures that SQL business queries are based on a clean product-demand data foundation rather than mixed transaction records.

## 2.3 Load Clean Sales into SQLite

The cleaned product sales table is loaded into a local SQLite database.

This allows SQL queries to be used for business reporting, aggregation, and product-level analysis.

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

# Define project paths
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

processed_dir = project_root / "data" / "processed"
database_dir = project_root / "database"
outputs_dir = project_root / "outputs"

database_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

# Define file paths
clean_sales_path = processed_dir / "clean_sales.csv"
db_path = database_dir / "ecommerce_inventory.db"

# Load cleaned product sales data
sales = pd.read_csv(clean_sales_path)

# Create SQLite connection
conn = sqlite3.connect(db_path)

# Write clean sales data into SQLite
sales.to_sql("clean_sales", conn, if_exists="replace", index=False)
conn.commit()

print("Clean product sales rows loaded:", len(sales))
print("Database path:", db_path)

/var/folders/p1/npj8kxr13mn73tw6fz0ytw2m0000gn/T/ipykernel_23137/3386200302.py:20: DtypeWarning: Columns (0: invoice_no) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv(clean_sales_path)


Clean product sales rows loaded: 522716
Database path: /Users/yeternalh/ecommerce-inventory-warehouse-allocation/database/ecommerce_inventory.db


## 2.4 Database Connection Check

Before running business queries, I verify that the cleaned sales table has been successfully loaded into SQLite.

This confirms that the SQL analysis is connected to the cleaned product sales data generated from the previous notebook.

In [2]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,clean_sales


In [3]:
row_count = pd.read_sql(
    "SELECT COUNT(*) AS row_count FROM clean_sales;",
    conn
)

row_count

,row_count
0,522716


## 2.5 Data Scope Validation

Since non-product transaction lines were excluded in the data cleaning stage, I validate that these stock codes are no longer included in the SQL analysis table.

This helps confirm that the downstream business queries focus only on physical product sales.

In [4]:
non_product_check_query = """
SELECT
    stock_code,
    COUNT(*) AS row_count,
    ROUND(SUM(revenue), 2) AS total_revenue
FROM clean_sales
WHERE stock_code IN ('POST', 'DOT', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'D', 'M')
GROUP BY stock_code;
"""

non_product_check = pd.read_sql(non_product_check_query, conn)
non_product_check

,stock_code,row_count,total_revenue


In [5]:
print("Non-product stock codes remaining in clean_sales:", len(non_product_check))

Non-product stock codes remaining in clean_sales: 0


## 2.6 Top SKU Revenue Contribution

This query identifies the SKUs with the highest revenue contribution.

From a management perspective, these SKUs should receive higher attention in inventory planning because stockouts in high-revenue products may directly affect sales performance.

In [6]:
top_sku_query = """
SELECT
    stock_code,
    description,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    ROUND(AVG(unit_price), 2) AS avg_unit_price
FROM clean_sales
GROUP BY stock_code, description
ORDER BY total_revenue DESC
LIMIT 20;
"""

top_sku = pd.read_sql(top_sku_query, conn)
top_sku

,stock_code,description,total_units,total_revenue,order_count,avg_unit_price
0,22423,REGENCY CAKESTAND 3 TIER,13851,174156.54,1988,13.98
1,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1,2.08
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37580,104284.24,2189,3.12
3,47566,PARTY BUNTING,18283,99445.23,1685,5.80
4,85099B,JUMBO BAG RED RETROSPOT,48371,94159.81,2089,2.49
5,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92,247,1.47
6,23084,RABBIT NIGHT LIGHT,30739,66870.03,994,2.39
7,22086,PAPER CHAIN KIT 50'S CHRISTMAS,19329,64875.59,1160,3.36
8,84879,ASSORTED COLOUR BIRD ORNAMENT,36362,58927.62,1455,1.72
9,79321,CHILLI LIGHTS,10302,54096.36,661,6.80


In [7]:
top_sku.to_csv(outputs_dir / "top_sku_revenue_contribution.csv", index=False)

## 2.7 Top SKU Unit Contribution

Revenue contribution and unit sales are not always the same.

This query identifies SKUs with the highest sales volume. High-volume SKUs may require more stable replenishment planning even if their unit price is relatively low.

In [8]:
top_units_query = """
SELECT
    stock_code,
    description,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    ROUND(AVG(unit_price), 2) AS avg_unit_price
FROM clean_sales
GROUP BY stock_code, description
ORDER BY total_units DESC
LIMIT 20;
"""

top_units = pd.read_sql(top_units_query, conn)
top_units

,stock_code,description,total_units,total_revenue,order_count,avg_unit_price
0,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.60,1,2.08
1,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92,247,1.47
2,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,54951,13814.01,535,0.32
3,85099B,JUMBO BAG RED RETROSPOT,48371,94159.81,2089,2.49
4,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37580,104284.24,2189,3.12
5,22197,POPCORN HOLDER,36749,34288.67,803,1.02
6,21212,PACK OF 72 RETROSPOT CAKE CASES,36396,21246.45,1320,0.76
7,84879,ASSORTED COLOUR BIRD ORNAMENT,36362,58927.62,1455,1.72
8,23084,RABBIT NIGHT LIGHT,30739,66870.03,994,2.39
9,22492,MINI PAINT SET VINTAGE,26633,16937.82,380,0.79


In [9]:
top_units.to_csv(outputs_dir / "top_sku_unit_contribution.csv", index=False)

## 2.8 Long-Tail SKU Identification

Long-tail SKUs usually sell in low volume and may not need large local warehouse inventory.

Identifying long-tail products helps management reduce overstock risk and avoid using warehouse space for slow-moving items.

In [10]:
long_tail_query = """
SELECT
    stock_code,
    description,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    COUNT(DISTINCT invoice_month) AS active_months,
    ROUND(AVG(unit_price), 2) AS avg_unit_price
FROM clean_sales
GROUP BY stock_code, description
HAVING total_units <= 20
ORDER BY total_units ASC, total_revenue ASC
LIMIT 50;
"""

long_tail_skus = pd.read_sql(long_tail_query, conn)
long_tail_skus

,stock_code,description,total_units,total_revenue,order_count,active_months,avg_unit_price
0,84227,HEN HOUSE W CHICK IN NEST,1,0.42,1,1,0.42
1,23366,SET 12 COLOURING PENCILS DOILEY,1,0.65,1,1,0.65
2,51014c,"FEATHER PEN,COAL BLACK",1,0.83,1,1,0.83
3,90084,PINK CRYSTAL GUITAR PHONE CHARM,1,0.85,1,1,0.85
4,21009,ETCHED GLASS STAR TREE DECORATION,1,1.25,1,1,1.25
5,23370,SET 36 COLOURING PENCILS DOILEY,1,1.25,1,1,1.25
6,35597A,DUSTY PINK CHRISTMAS TREE 30CM,1,1.25,1,1,1.25
7,35597B,BLACKCHRISTMAS TREE 30CM,1,1.25,1,1,1.25
8,37461,FUNKY MONKEY MUG,1,1.25,1,1,1.25
9,84569C,PACK 4 FLOWER/BUTTERFLY PATCHES,1,1.25,1,1,1.25


In [11]:
long_tail_skus.to_csv(outputs_dir / "long_tail_skus.csv", index=False)

## 2.9 Country-Level Demand Summary

The dataset contains country-level demand information.

Although this project later focuses on inventory and warehouse logic, country-level demand analysis demonstrates how regional sales distribution can support warehouse allocation and fulfillment planning.

In [12]:
country_demand_query = """
SELECT
    country,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    COUNT(DISTINCT stock_code) AS unique_skus
FROM clean_sales
GROUP BY country
ORDER BY total_revenue DESC;
"""

country_demand = pd.read_sql(country_demand_query, conn)
country_demand.head(20)

,country,total_units,total_revenue,order_count,unique_skus
0,United Kingdom,4639181,8737732.14,17904,3911
1,Netherlands,200258,283889.34,93,781
2,EIRE,147002,276090.86,284,1967
3,Germany,118032,205381.15,443,1662
4,France,111229,184679.00,383,1540
5,Australia,83890,138103.81,56,598
6,Spain,27724,55706.56,88,1090
7,Switzerland,30515,53065.60,50,977
8,Japan,26016,37416.37,19,215
9,Belgium,22962,36927.34,98,776


In [13]:
country_demand.to_csv(outputs_dir / "country_demand_summary.csv", index=False)

## 2.10 Monthly Sales Trend

This query summarizes total monthly sales volume and revenue.

The output will be used to understand overall sales seasonality and demand patterns before moving to SKU-level classification and replenishment planning.

In [14]:
monthly_sales_query = """
SELECT
    invoice_month,
    SUM(quantity) AS total_units,
    ROUND(SUM(revenue), 2) AS total_revenue,
    COUNT(DISTINCT invoice_no) AS order_count,
    COUNT(DISTINCT stock_code) AS unique_skus
FROM clean_sales
GROUP BY invoice_month
ORDER BY invoice_month;
"""

monthly_sales = pd.read_sql(monthly_sales_query, conn)
monthly_sales

,invoice_month,total_units,total_revenue,order_count,unique_skus
0,2010-12,357542,776317.50,1551,2783
1,2011-01,386745,670639.46,1081,2569
2,2011-02,282635,508081.54,1093,2395
3,2011-03,376198,690591.84,1440,2499
4,2011-04,307665,515899.66,1236,2455
5,2011-05,394667,740472.33,1668,2454
6,2011-06,388141,738233.99,1525,2622
7,2011-07,399303,688802.67,1452,2667
8,2011-08,420710,735770.22,1341,2593
9,2011-09,568720,1029245.38,1819,2729


In [15]:
monthly_sales.to_csv(outputs_dir / "monthly_sales_trend.csv", index=False)

## 2.11 Output Summary

This notebook generates five SQL-based business summary outputs:

1. `top_sku_revenue_contribution.csv`: top SKUs by revenue contribution.
2. `top_sku_unit_contribution.csv`: top SKUs by sales volume.
3. `long_tail_skus.csv`: low-volume SKUs that may require cautious stocking.
4. `country_demand_summary.csv`: country-level demand and revenue summary.
5. `monthly_sales_trend.csv`: monthly sales volume and revenue trend.

These outputs are designed to support SKU classification, inventory planning, and warehouse allocation analysis in later notebooks.

## 2.12 Management Implication

The SQL queries convert cleaned product sales data into management-facing business reporting tables.

These outputs help management understand:

1. Which SKUs generate the most revenue.
2. Which SKUs sell the most units.
3. Which SKUs may be long-tail or slow-moving.
4. Which countries or regions contribute the most demand.
5. How overall sales volume and revenue change over time.

The next step is to classify SKUs by sales contribution and demand volatility, using the SQL outputs and monthly SKU-level sales data as analytical references.

In [16]:
print("===== 02 SQL Business Queries Summary =====")
print("Clean product sales rows loaded:", len(sales))
print("Non-product stock codes remaining in clean_sales:", len(non_product_check))
print("Top SKU revenue table shape:", top_sku.shape)
print("Top SKU unit table shape:", top_units.shape)
print("Long tail SKU table shape:", long_tail_skus.shape)
print("Country demand table shape:", country_demand.shape)
print("Monthly sales trend shape:", monthly_sales.shape)

print("\nOutput files:")
print("- outputs/top_sku_revenue_contribution.csv")
print("- outputs/top_sku_unit_contribution.csv")
print("- outputs/long_tail_skus.csv")
print("- outputs/country_demand_summary.csv")
print("- outputs/monthly_sales_trend.csv")

===== 02 SQL Business Queries Summary =====
Clean product sales rows loaded: 522716
Non-product stock codes remaining in clean_sales: 0
Top SKU revenue table shape: (20, 6)
Top SKU unit table shape: (20, 6)
Long tail SKU table shape: (50, 7)
Country demand table shape: (38, 5)
Monthly sales trend shape: (13, 5)

Output files:
- outputs/top_sku_revenue_contribution.csv
- outputs/top_sku_unit_contribution.csv
- outputs/long_tail_skus.csv
- outputs/country_demand_summary.csv
- outputs/monthly_sales_trend.csv
